# Train Next-Action Models With Category Feature

Notebook này sửa 3 vấn đề của pipeline cũ:

- label phải là **hành động kế tiếp**, không phải action cuối của chính input
- cột `category` phải được đưa vào feature train
- so sánh RNN, LSTM, biLSTM trên cùng một pipeline đúng


In [ ]:
import os
import random

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Input, LSTM, SimpleRNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
MAX_LEN = 10

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
df = pd.read_csv("data_user500.csv")
df.head()


In [ ]:
action_encoder = LabelEncoder()
category_encoder = LabelEncoder()

df["action_encoded"] = action_encoder.fit_transform(df["action"])
df["category_encoded"] = category_encoder.fit_transform(df["category"])

print("Action classes:", list(action_encoder.classes_))
print("Category classes:", list(category_encoder.classes_))
print("Action distribution:")
print(df["action"].value_counts(normalize=True).round(4))


In [ ]:
action_sequences = []
category_sequences = []
labels = []

for _, group in df.groupby("user_id"):
    group = group.sort_values("timestamp")
    actions = group["action_encoded"].tolist()
    categories = group["category_encoded"].tolist()

    for i in range(1, len(group)):
        action_sequences.append(actions[:i])
        category_sequences.append(categories[:i])
        labels.append(actions[i])


In [ ]:
X_action = pad_sequences(action_sequences, maxlen=MAX_LEN, padding="pre", truncating="pre")
X_category = pad_sequences(category_sequences, maxlen=MAX_LEN, padding="pre", truncating="pre")
X = np.stack([X_action, X_category], axis=-1).astype("float32")
y = np.array(labels, dtype="int32")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

majority_baseline = pd.Series(y_test).value_counts(normalize=True).max()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Majority baseline:", round(float(majority_baseline), 4))


In [ ]:
def build_model(kind, input_shape=(MAX_LEN, 2), num_classes=len(action_encoder.classes_)):
    if kind == "RNN":
        backbone = SimpleRNN(64)
    elif kind == "LSTM":
        backbone = LSTM(64)
    elif kind == "biLSTM":
        backbone = Bidirectional(LSTM(64))
    else:
        raise ValueError(f"Unsupported model kind: {kind}")

    model = Sequential([
        Input(shape=input_shape),
        backbone,
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model


In [ ]:
callbacks = [EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
models = {}
histories = {}

for kind in ["RNN", "LSTM", "biLSTM"]:
    model = build_model(kind)
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=12,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )
    models[kind] = model
    histories[kind] = history


In [ ]:
results = {}
for kind, model in models.items():
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results[kind] = {
        "best_val_accuracy": float(max(histories[kind].history["val_accuracy"])),
        "test_accuracy": float(test_acc),
    }

results


In [ ]:
plt.figure(figsize=(12, 5))
for kind, history in histories.items():
    plt.plot(history.history["accuracy"], label=f"{kind} train")
    plt.plot(history.history["val_accuracy"], linestyle="--", label=f"{kind} val")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
best_scores = {kind: metric["best_val_accuracy"] for kind, metric in results.items()}

plt.figure(figsize=(7, 4))
plt.bar(best_scores.keys(), best_scores.values(), color=["#2563eb", "#16a34a", "#ea580c"])
plt.title("Best Validation Accuracy Comparison")
plt.ylabel("Best val_accuracy")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
models["RNN"].save("model_rnn.h5")
models["LSTM"].save("model_lstm.h5")
models["biLSTM"].save("model_bilstm.h5")
